# 02 — ベクトル、単位、座標系

四足制御で最も多い不具合は、値そのものよりshape・脚順・座標系の取り違えです。

**前提**: `01_environment_and_source_map.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: 各章を同じ作業ディレクトリと依存関係で再実行するには、リポジトリの基準パスと外部実装の場所を最初に固定する必要がある。
# 目的: Quadruped-PyMPCをimport可能にし、acadosとヘッドレスMuJoCoの実行環境を後続セルへ引き渡す。
# ファイルシステム上の基準パスを型安全に扱うためPathを読み込む。
from pathlib import Path
# 環境変数の設定とPythonのモジュール探索パス更新に必要な標準ライブラリを読み込む。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化し、教材全体の基準候補とする。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけ、リポジトリ直下へ基準を1階層戻す。
if ROOT.name == "notebook_pympc":
    # externalディレクトリを参照できるリポジトリ直下へROOTを合わせる。
    ROOT = ROOT.parent
# 上流制御実装が置かれたQuadruped-PyMPCの絶対パスを構成する。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動位置のまま進まず、依存リポジトリの欠落を具体的なパス付きで検出する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同じパスを重複登録せず、まだimport探索対象でない場合だけ追加する。
if str(PYMPC_ROOT) not in sys.path:
    # ローカルのquadruped_pympcパッケージを通常のimport文で読めるよう探索順の先頭へ置く。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物・共有資源の基準位置を未設定時だけ登録し、利用者の明示設定は上書きしない。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画可能にするため、未設定時のOpenGL backendをEGLにする。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実際に採用されたworkspace基準を表示し、相対パス問題を診断できるようにする。
print("workspace :", ROOT)
# import対象となる上流実装の場所を表示し、参照しているコード版を確認可能にする。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 3つの座標系

- World \(W\): 重力と地面に固定
- Base \(B\): 胴体と一緒に回転
- Heading \(H\): yawだけ胴体に追従し、roll/pitchは除く

回転行列を \(R_{A\leftarrow B}\) と書くと、

\[
{}^A v = R_{A\leftarrow B}\,{}^B v,\qquad
R_{B\leftarrow A}=R_{A\leftarrow B}^{T}
\]

位置は原点の差も必要です。速度ベクトルと位置ベクトルを同じ式で変換してはいけません。

In [2]:
# 背景: World座標Wとyaw追従のHeading座標Hを混同すると、同じ速度指令でも進行方向が回転して解釈される。
# 目的: 30度のyawで速度ベクトルをW→H→Wと変換し、回転行列の向き・shape・直交性を数値確認する。
# 2次元ベクトルと回転行列の構成・積・比較にNumPyを使う。
import numpy as np
# 人が読みやすい30度を、三角関数が要求するrad単位へ変換する。
yaw = np.deg2rad(30)
# ^H v = R_{H←W} ^W vを満たす2×2のyaw回転行列を構成する。
R_W2H = np.array([[np.cos(yaw), np.sin(yaw)],
                  [-np.sin(yaw), np.cos(yaw)]])
# World座標でx方向へ1 m/s進むshape (2,)の速度ベクトルを用意する。
v_W = np.array([1.0, 0.0])
# 2×2行列と(2,)ベクトルを掛け、Heading座標の速度[m/s]へ変換する。
v_H = R_W2H @ v_W
# 変換前のWorld速度と単位を表示して比較基準を明示する。
print("v_W:", v_W, "m/s")
# Heading速度を小数3桁で表示し、30度回転による成分分解を確認する。
print("v_H:", np.round(v_H, 3), "m/s")
# 逆変換R_{W←H}=R_{H←W}^Tで元のWorld速度へ戻ることを表示する。
print("round trip:", np.round(R_W2H.T @ v_H, 12))
# R^T R=Iを検査し、転置を逆行列として使える直交回転であることを保証する。
assert np.allclose(R_W2H.T @ R_W2H, np.eye(2))

v_W: [1. 0.] m/s
v_H: [ 0.866 -0.5  ] m/s
round trip: [1. 0.]


## shape契約

標準SRBDでは、基本状態は24要素
`(COM位置3, 速度3, Euler角3, 角速度3, 足位置12)`。
実装は積分状態6要素も確保し、実際の `states_dim` は30です。
入力は `(足速度12, 床反力12)` の24要素です。
脚順は `FL, FR, RL, RR` です。

In [3]:
# 背景: SRBDのベクトルを意味のない30要素として扱うと、脚順や状態block境界のずれを発見しにくい。
# 目的: 状態を物理量ごとの次元へ分解し、脚順FL,FR,RL,RRとstate=30・input=24のshape契約を検算する。
# 4脚ベクトルを連結するときに上流実装が採用する固定順序を定義する。
LEG_ORDER = ("FL", "FR", "RL", "RR")
# 状態xの各block名と要素数を対応づけ、3+3+3+3+12+6=30を明示する。
blocks = {"position": 3, "velocity": 3, "rpy": 3, "omega": 3, "feet": 12, "integrals": 6}
# block次元の総和を表示し、CasADi stateがshape (30,)になる根拠を確認する。
print("state dimension:", sum(blocks.values()))
# 足速度4脚×3軸とGRF4脚×3軸を連結したinput次元24を表示する。
print("input dimension:", 12 + 12)
# docstringの古い29次元表記に依存せず、現行block構成が30要素であることを検査する。
assert sum(blocks.values()) == 30

state dimension: 30
input dimension: 24


**注意**: 上流 `forward_dynamics` のdocstringには29次元と残っていますが、
コードで連結される状態は30次元です。説明とコードが食い違う場合は、
シンボルの `size()` と実行結果を正にします。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。